In [4]:
%load_ext autoreload
%autoreload 2

import plotly.express as px
import torch
import torch.optim as optim
from models.custom_net import Net
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from train import Trainer
from utils import get_datasets_and_loaders

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
classes = ("buildings", "forest", "glacier", "mountain", "sea", "street")

train_set, train_loader, valid_set, valid_loader = get_datasets_and_loaders()

In [7]:
image, label = next(iter(train_loader))
image = image[0:8]
image = image.permute(0, 2, 3, 1)
fig = px.imshow(image, facet_col=0, facet_col_spacing=0.01, height=300)
for i in range(image.shape[0]):
    fig.layout.annotations[i]["text"] = classes[label[i]]
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

In [ ]:
num_epochs = 20

model = Net()

loaders = {"train_loader": train_loader, "valid_loader": valid_loader}

optimizer = optim.Adam(params=model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, total_steps=len(train_loader) * num_epochs
)
optimization = {
    "criterion": nn.CrossEntropyLoss(),
    "optimizer": optimizer,
    "scheduler": scheduler,
}

metrics = MetricCollection(
    {
        "Accuracy": MulticlassAccuracy(num_classes=6, average="micro"),
        "Precision": MulticlassPrecision(num_classes=6, average="macro"),
        "Recall": MulticlassRecall(num_classes=6, average="macro"),
        "F1": MulticlassF1Score(num_classes=6, average="macro"),
    }
)
conf_matrix = MulticlassConfusionMatrix(num_classes=6)
log = {"metrics": metrics, "conf_matrix": conf_matrix, "writer": SummaryWriter()}

config = {
    "checkpoint_path": "checkpoints/",
    "classes": classes,
}

trainer = Trainer(model, loaders, optimization, log, config)

In [ ]:
trainer.fit(
    num_epochs=num_epochs,
    start_epoch=trainer.load_checkpoint(path="checkpoints/last_model_checkpoint.pth"),
)

## TODO

1. Оценить производительность модели
2. Импортировать ResNet50 и сделать fine-tuning